# 插入数据到 Collection

## 用途
教学演示 - 如何向 Milvus Collection 插入数据

## 核心概念
- 数据插入方式（单条/批量）
- 向量数据 + 标量数据同时插入
- 自增 ID vs 手动指定 ID
- Embedding 模型生成向量（文本/音频/图像）

## 运行前检查
1. 已安装依赖：`pip install -r requirements.txt`
2. 已完成 `01_connect_milvus.py` 和 `02_create_collection.py` 的学习
3. 确保 Collection 已创建（运行本脚本会自动创建）
4. 请确保 `.env` 文件中配置了 `ALIYUN_API_KEY`（用于真实 Embedding 生成）

In [ ]:
# 导入 Milvus 配置（优先使用 Docker Milvus）
import sys
import os
from dotenv import load_dotenv
load_dotenv()

from rag_examples.milvus_config import MILVUS_URI

## 第一部分：理解数据插入

### 数据插入的核心要点

**插入方式**
1. 单条插入：一次插入一条数据（适合实时写入）
2. 批量插入：一次插入多条数据（适合批量导入，性能更好）

### 数据格式
Milvus 插入的数据是一个字典列表，每个字典代表一行：
```python
[
  {"content": "文本 1", "embedding": [0.1, 0.2, ...], "category": "A"},
  {"content": "文本 2", "embedding": [0.3, 0.4, ...], "category": "B"},
]
```

### ID 说明
- 自增 ID（auto_id=True）：无需指定 id，Milvus 自动生成
- 手动指定 ID：需要确保 ID 唯一

### 注意事项
1. 向量维度必须与 Collection 定义一致
2. 标量字段的类型必须匹配
3. 批量插入性能优于单条插入

## 示例 1: 理解 Embedding（嵌入）

In [ ]:
def explain_embedding():
    """
    理解什么是 Embedding（嵌入）

    Embedding 是将文本、图像、音频等数据转换为向量的过程
    这些向量可以理解为数据的"数学指纹"
    """
    print("=" * 60)
    print("示例 1: 理解 Embedding（嵌入）")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 什么是 Embedding（嵌入）？                               │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 📊 定义                                                  │
│    Embedding = 将非结构化数据转换为向量（数字列表）     │
│    相似的内容 → 向量在空间中距离相近                    │
│                                                         │
│ 💡 生活化比喻                                            │
│    Embedding = "给数据分配坐标"                          │
│                                                         │
│    图书馆里，相似主题的书放在一起：                       │
│    - AI 书籍 → 坐标 [0.8, 0.2, 0.9, ...]                 │
│    - 烹饪书籍 → 坐标 [0.1, 0.9, 0.3, ...]                │
│    - 历史书籍 → 坐标 [0.3, 0.7, 0.2, ...]                │
│                                                         │
│    语义相近的书，坐标也相近！                            │
│                                                         │
│ 📦 Embedding 模型类型                                    │
│    1. 文本 Embedding 模型                                │
│       - text-embedding-v4 (阿里云百炼)                  │
│       - text-embedding-v3 (阿里云百炼)                  │
│       - BGE-M3 (开源)                                   │
│       - m3e-base (开源，中文优化)                        │
│                                                         │
│    2. 图像 Embedding 模型                                │
│       - ResNet50                                        │
│       - ViT (Vision Transformer)                        │
│       - CLIP (图文双模态)                               │
│                                                         │
│    3. 音频 Embedding 模型                                │
│       - VGGish (Google)                                 │
│       - YAMNet                                          │
│       - Whisper Encoder                                 │
│                                                         │
│ 🔑 为什么需要 Embedding？                                │
│    - 计算机只能理解数字                                  │
│    - 向量空间中的距离 = 语义相似度                       │
│    - 可以用数学方法计算"相似"                            │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 Embedding 示例（文本 → 向量）:

   "我喜欢猫" → [0.12, -0.34, 0.56, 0.78, ...]  (1024 维)
   "我讨厌狗" → [0.10, -0.32, 0.54, 0.76, ...]  (1024 维)
                      ↑
              向量非常接近！

   "今天天气真好" → [0.89, 0.45, -0.12, 0.33, ...] (1024 维)
                      ↑
              向量距离较远
""")

In [ ]:
# 运行示例 1
explain_embedding()

## 示例 2: 使用真实 Embedding 模型生成向量

使用阿里云百炼 DashScope API 生成真实的 Embedding 向量。

In [ ]:
def generate_embedding_with_llm(texts, model="text-embedding-v4", api_key=None):
    """
    使用阿里云百炼 DashScope API 生成真实的 Embedding 向量

    支持的模型:
    - text-embedding-v4: 最新模型，支持 1024 维
    - text-embedding-v3: 支持多语言，支持 1024/1536/2048 维

    参数:
        texts: 文本列表或单个文本
        model: Embedding 模型名称
        api_key: API Key，默认从环境变量读取
    返回:
        向量列表（每个向量是浮点数列表）
    """
    from openai import OpenAI

    # 获取 API Key
    if api_key is None:
        api_key = os.getenv("DASHSCOPE_API_KEY") or os.getenv("ALIYUN_API_KEY")
        if not api_key:
            raise ValueError("未找到 API Key，请设置环境变量 DASHSCOPE_API_KEY 或 ALIYUN_API_KEY")

    # 初始化客户端（使用阿里云百炼兼容模式）
    client = OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    # 支持单个文本或列表
    if isinstance(texts, str):
        texts = [texts]
        single_input = True
    else:
        single_input = False

    print(f"调用 Embedding 模型：{model}")
    print(f"输入文本数量：{len(texts)}")

    try:
        # 调用 Embedding API
        # 注意：API 一次最多处理 25 条文本
        all_embeddings = []
        batch_size = 25

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            response = client.embeddings.create(
                model=model,
                input=batch,
                encoding_format="float"
            )

            # 提取向量数据
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)

            print(f"  已处理批次 {i//batch_size + 1}: {len(batch_embeddings)} 条")

        # 获取向量维度
        if all_embeddings:
            print(f"生成向量维度：{len(all_embeddings[0])} 维")

        return all_embeddings[0] if single_input else all_embeddings

    except Exception as e:
        print(f"Embedding API 调用失败：{e}")
        print("退化为模拟 Embedding 生成...")
        # 退化方案：使用模拟 Embedding
        return generate_mock_embeddings(texts if not single_input else texts[0])

In [ ]:
def demo_embedding_generation():
    """
    演示真实 Embedding 生成
    """
    print("=" * 60)
    print("示例 2: 使用真实 Embedding 模型生成向量")
    print("=" * 60)

    # 测试文本
    texts = [
        "人工智能是模拟人类智能的计算机科学",
        "机器学习通过训练数据让计算机自动学习",
        "深度学习使用神经网络模拟人脑",
    ]

    print(f"\n输入文本:\n")
    for i, text in enumerate(texts):
        print(f"  {i+1}. {text}")

    # 尝试使用真实 API
    try:
        api_key = os.getenv("DASHSCOPE_API_KEY") or os.getenv("ALIYUN_API_KEY")
        if api_key:
            print(f"\n检测到 API Key，正在调用真实 Embedding 模型...\n")
            embeddings = generate_embedding_with_llm(texts, api_key=api_key)

            print(f"\n生成的 Embedding 向量预览:")
            for i, emb in enumerate(embeddings):
                print(f"  文本{i+1} → [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, ...] ({len(emb)}维)")
        else:
            print("\n未检测到 API Key，使用模拟 Embedding 演示\n")
            embeddings = generate_mock_embeddings(texts)
            print(f"模拟 Embedding 向量预览:")
            for i, emb in enumerate(embeddings):
                print(f"  文本{i+1} → [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, ...] ({len(emb)}维)")
    except Exception as e:
        print(f"\n调用失败：{e}")
        print("\n使用模拟 Embedding 演示\n")
        embeddings = generate_mock_embeddings(texts)
        for i, emb in enumerate(embeddings):
            print(f"  文本{i+1} → [{emb[0]:.4f}, {emb[1]:.4f}, {emb[2]:.4f}, ...] ({len(emb)}维)")

In [ ]:
# 运行示例 2
demo_embedding_generation()

## 示例 3: 模拟 Embedding 数据

⚠️ **重要提示**：此函数仅用于 API 不可用时的降级方案，实际项目必须使用真实的 Embedding 模型！

In [ ]:
def generate_mock_embeddings(texts, dim=768):
    """
    模拟 Embedding 生成的向量数据

    仅用于教学演示和 API 不可用时的降级方案
    实际项目必须使用真实的 Embedding 模型！

    参数:
        texts: 文本列表
        dim: 向量维度
    返回:
        向量列表（每个向量是 dim 维的浮点数列表）
    """
    import random
    random.seed(42)  # 固定随机种子，确保可重复运行

    # 支持单个文本或列表
    if isinstance(texts, str):
        texts = [texts]

    vectors = []
    for text in texts:
        # 模拟：生成随机向量
        # ⚠️ 注意：实际项目中必须使用真实 Embedding 模型！
        # 例如：
        #   from sentence_transformers import SentenceTransformer
        #   model = SentenceTransformer('BAAI/bge-m3')
        #   vectors = model.encode(texts).tolist()
        vector = [random.random() for _ in range(dim)]
        vectors.append(vector)

    return vectors[0] if len(texts) == 1 else vectors

In [ ]:
# 测试模拟 Embedding
test_embeddings = generate_mock_embeddings(["测试文本1", "测试文本2"], dim=10)
print(f"生成的模拟向量: {test_embeddings}")

## 示例 4: 准备测试数据

In [ ]:
def prepare_test_data():
    """
    准备测试用的文档数据

    包含：文本内容、类别、向量
    """
    # 测试文档列表
    documents = [
        {
            "content": "人工智能（AI）是模拟人类智能的计算机科学领域，包括机器学习、深度学习、自然语言处理等技术。",
            "title": "人工智能简介",
            "category": "AI",
            "views": 1000
        },
        {
            "content": "机器学习是人工智能的分支，通过训练数据让计算机自动学习规律，无需显式编程。常见算法包括决策树、神经网络、支持向量机等。",
            "title": "机器学习基础",
            "category": "AI",
            "views": 800
        },
        {
            "content": "深度学习使用多层神经网络模拟人脑，在图像识别、语音识别、自然语言处理等领域取得突破性进展。",
            "title": "深度学习入门",
            "category": "AI",
            "views": 1200
        },
        {
            "content": "RAG（检索增强生成）结合检索和生成技术，先检索相关知识库，再让大语言模型基于检索结果生成答案。",
            "title": "RAG 技术解析",
            "category": "LLM",
            "views": 600
        },
        {
            "content": "Milvus 是一个开源的向量数据库，专门用于存储和搜索向量数据，支持亿级向量毫秒级检索。",
            "title": "Milvus 向量数据库",
            "category": "Database",
            "views": 500
        },
    ]

    return documents

In [ ]:
# 运行示例 4
documents = prepare_test_data()
print(f"准备测试数据: {len(documents)} 条")
for doc in documents:
    print(f"  - {doc['title']}")

## 示例 5: 单条插入数据

演示单条插入数据，适合实时写入场景，但性能较低。

In [ ]:
def insert_single_data():
    """
    演示单条插入数据

    适合实时写入场景，但性能较低

    ⚠️ 注意：实际项目中请使用真实的 Embedding 模型
              本示例使用 mock 向量仅用于演示插入流程
    """
    print("=" * 60)
    print("示例 5: 单条插入数据")
    print("=" * 60)

    from pymilvus import MilvusClient

    # 连接 Milvus
    client = MilvusClient(uri=MILVUS_URI)

    # Collection 名称
    collection_name = "simple_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 创建 Collection（简单版本，768 维）
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE"
    )

    # 准备测试数据
    documents = prepare_test_data()
    vectors = generate_mock_embeddings([d["content"] for d in documents])

    print(f"准备插入 {len(documents)} 条数据...\n")

    # 逐条插入
    for i, doc in enumerate(documents):
        data = {
            "content": doc["content"],
            "vector": vectors[i]  # 使用 vector 而不是 embedding
        }

        # 插入单条数据
        result = client.insert(
            collection_name=collection_name,
            data=data
        )

        print(f"✓ 插入第 {i+1} 条：{doc['title'][:10]}...")
        print(f"  返回结果：{result}")

    print()

    # 统计行数
    stats = client.get_collection_stats(collection_name)
    print(f"插入完成，总行数：{stats.get('row_count', 0)}")

    return client, collection_name

In [ ]:
# 运行示例 5
client, collection_name = insert_single_data()

## 示例 6: 批量插入数据（推荐）

推荐方式：一次性插入多条数据，性能更好。

In [ ]:
def insert_batch_data():
    """
    演示批量插入数据

    推荐方式：一次性插入多条数据，性能更好

    ⚠️ 注意：实际项目中请使用真实的 Embedding 模型
              本示例使用 mock 向量仅用于演示插入流程
    """
    print("=" * 60)
    print("示例 6: 批量插入数据（推荐）")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "simple_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 创建 Collection
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=True,
        metric_type="COSINE"
    )

    # 准备测试数据
    documents = prepare_test_data()
    texts = [d["content"] for d in documents]
    vectors = generate_mock_embeddings(texts)

    print(f"准备批量插入 {len(documents)} 条数据...\n")

    # 构造批量插入的数据格式
    # 方式 1：字典列表（推荐）
    data_to_insert = []
    for i, doc in enumerate(documents):
        data_to_insert.append({
            "content": doc["content"],
            "vector": vectors[i],  # 使用 vector 而不是 embedding
            "title": doc["title"]
        })

    # 批量插入
    result = client.insert(
        collection_name=collection_name,
        data=data_to_insert
    )

    print(f"✓ 批量插入完成")
    print(f"  插入数量：{result.get('insert_count', len(data_to_insert))}")
    print(f"  返回 ID 数量：{len(result.get('ids', []))}")

    # 统计行数
    stats = client.get_collection_stats(collection_name)
    print(f"\n总行数：{stats.get('row_count', 0)}")

    return client, collection_name

In [ ]:
# 运行示例 6
client, collection_name = insert_batch_data()

## 示例 7: 自定义字段 Collection 插入

演示如何向包含多个标量字段的 Collection 插入数据。

In [ ]:
def insert_with_custom_fields():
    """
    向自定义字段的 Collection 插入数据

    演示如何插入包含多个标量字段的数据

    ⚠️ 注意：实际项目中请使用真实的 Embedding 模型
              本示例使用 mock 向量仅用于演示插入流程
    """
    print("=" * 60)
    print("示例 7: 自定义字段 Collection 插入")
    print("=" * 60)

    from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "custom_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 定义字段
    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="content", dtype=DataType.VARCHAR, max_length=65535),
        FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=256),
        FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=64),
        FieldSchema(name="views", dtype=DataType.INT64),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=768),
    ]
    schema = CollectionSchema(fields=fields)

    # 创建 Collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        metric_type="COSINE"
    )

    # 准备数据
    documents = prepare_test_data()
    texts = [d["content"] for d in documents]
    vectors = generate_mock_embeddings(texts)

    print(f"准备插入带完整字段的数据...\n")

    # 构造数据（包含所有字段）
    data_to_insert = []
    for i, doc in enumerate(documents):
        data_to_insert.append({
            "content": doc["content"],
            "title": doc["title"],
            "category": doc["category"],
            "views": doc["views"],
            "vector": vectors[i]  # 使用 vector 而不是 embedding
        })

    # 批量插入
    result = client.insert(
        collection_name=collection_name,
        data=data_to_insert
    )

    print(f"✓ 插入完成")
    print(f"  插入数量：{result.get('insert_count', 0)}")

    # 验证数据：查询前 3 条
    # Milvus v2.5.0+: 查询前需要先加载集合（索引已存在则无需创建）
    print("\n检查索引...")
    index_list = client.list_indexes(collection_name=collection_name)
    if not index_list:
        from pymilvus.milvus_client import IndexParams
        index_params = IndexParams()
        index_params.add_index(field_name="vector", index_type="FLAT", metric_type="COSINE")
        client.create_index(
            collection_name=collection_name,
            index_params=index_params
        )
        print("索引已创建")
    else:
        print("索引已存在")

    print("加载集合...")
    client.load_collection(collection_name=collection_name)

    print("验证数据（查询前 3 条）：")
    res = client.query(
        collection_name=collection_name,
        filter="",
        output_fields=["title", "category", "views"],
        limit=3
    )

    for i, item in enumerate(res):
        print(f"  {i+1}. {item['title']} | 类别：{item['category']} | 浏览：{item['views']}")

    return client, collection_name

In [ ]:
# 运行示例 7
client, collection_name = insert_with_custom_fields()

## 示例 8: 手动指定 ID 插入

适用于需要同步外部 ID 的场景。

In [ ]:
def insert_with_custom_id():
    """
    演示手动指定 ID 的插入方式

    适用于需要同步外部 ID 的场景

    ⚠️ 注意：实际项目中请使用真实的 Embedding 模型
              本示例使用 mock 向量仅用于演示插入流程
    """
    print("=" * 60)
    print("示例 8: 手动指定 ID 插入")
    print("=" * 60)

    from pymilvus import MilvusClient

    client = MilvusClient(uri=MILVUS_URI)
    collection_name = "custom_id_docs"

    # 建表前先删表
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    # 创建 Collection（auto_id=False，需要手动指定 ID）
    client.create_collection(
        collection_name=collection_name,
        dimension=768,
        auto_id=False,  # 手动指定 ID
        metric_type="COSINE"
    )

    # 准备数据（手动指定 ID）
    documents = prepare_test_data()
    texts = [d["content"] for d in documents]
    vectors = generate_mock_embeddings(texts)

    print(f"手动指定 ID 插入数据...\n")

    # 构造数据（包含自定义 ID）
    data_to_insert = []
    custom_ids = [1001, 1002, 1003, 1004, 1005]  # 自定义 ID 列表

    for i, doc in enumerate(documents):
        data_to_insert.append({
            "id": custom_ids[i],  # 手动指定 ID
            "content": doc["content"],
            "vector": vectors[i]  # 使用 vector 而不是 embedding
        })

    # 插入数据
    result = client.insert(
        collection_name=collection_name,
        data=data_to_insert
    )

    print(f"✓ 插入完成")
    print(f"  返回的 ID：{result.get('ids', [])}")

    # 验证：通过 ID 查询特定数据
    print("\n检查索引...")
    index_list = client.list_indexes(collection_name=collection_name)
    if not index_list:
        from pymilvus.milvus_client import IndexParams
        index_params = IndexParams()
        index_params.add_index(field_name="vector", index_type="FLAT", metric_type="COSINE")
        client.create_index(
            collection_name=collection_name,
            index_params=index_params
        )
        print("索引已创建")
    else:
        print("索引已存在")

    print("加载集合...")
    client.load_collection(collection_name=collection_name)

    print("验证：通过 ID 查询 id=1003 的数据")
    res = client.query(
        collection_name=collection_name,
        filter="id == 1003",
        output_fields=["content"]
    )

    if res:
        print(f"  查询结果：{res[0]['content'][:30]}...")

    return client, collection_name

In [ ]:
# 运行示例 8
client, collection_name = insert_with_custom_id()

## 示例 9: 插入数据最佳实践

In [ ]:
def insert_best_practices():
    """
    插入数据的最佳实践建议

    总结实际项目中的经验和注意事项
    """
    print("=" * 60)
    print("示例 9: 插入数据最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 插入数据最佳实践                                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 批量插入优于单条插入                                  │
│    - 批量大小建议：100-1000 条/批                         │
│    - 太少：网络开销大                                   │
│    - 太多：内存占用高，失败重试成本大                   │
│                                                         │
│ 2. 向量维度必须匹配                                      │
│    - 插入前检查 Collection 的 dimension                   │
│    - 向量长度 != dimension 会报错                        │
│                                                         │
│ 3. 数据类型要一致                                        │
│    - VARCHAR 字段不能超过 max_length                     │
│    - INT64 字段必须是整数                               │
│                                                         │
│ 4. 错误处理                                              │
│    - 捕获异常并记录失败的 data                          │
│    - 实现重试机制                                       │
│                                                         │
│ 5. 性能优化                                              │
│    - 插入前不创建索引（插入完成后再建索引）             │
│    - 大量数据插入时考虑分区                             │
│                                                         │
│ 6. ID 选择策略                                           │
│    - 无需外部同步：用 auto_id=True（推荐）              │
│    - 需要同步外部 ID：用 auto_id=False                  │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 推荐代码模板:

```python
def insert_documents(documents, embeddings, collection_name):
    # 1. 准备数据
    data = [
        {"content": doc, "embedding": emb}
        for doc, emb in zip(documents, embeddings)
    ]

    # 2. 批量插入（建议每批 500 条）
    batch_size = 500
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        try:
            client.insert(collection_name, batch)
        except Exception as e:
            print(f"批次 {i} 插入失败：{e}")
            # 重试逻辑...
```
""")

In [ ]:
# 运行示例 9
insert_best_practices()